# Lesson 5 — Conditional edges: the agent loop

**Goal:** let the model *decide* whether it needs a tool. If it does, the
graph runs the tool and **loops back** to the model with the result. If it
doesn't, the graph finishes. This branching loop is the thing a straight
`prompt | model | parser` chain cannot do — and the reason LangGraph exists.

## The idea in one picture
```
          +-------------+
  START-> |  call_model | <----------+
          +-------------+            |
                 |                   | (tool result fed back)
        does it want a tool?         |
           /          \             |
        yes            no           |
         |              |           |
    +---------+      +-----+        |
    |  tools  |------+| END |       |
    +---------+       +-----+       |
         +---------------------------+
```
The diamond is a **conditional edge**: a function looks at the state and
returns the *name of the next node*. That's the only new concept here.

### The proxy parallel
Your proxy has no tool loop — it always does exactly one model call per
request (`main.py:chat_completions`). To add 'look something up, then answer'
today you'd hand-write nested `if`s and a `while` around the httpx call. The
conditional edge below *is* that control flow, but declared instead of coded.

In [ ]:
from typing import Annotated, TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

## 1. Define a tool

A *tool* is just a Python function the model is allowed to call. The `@tool`
decorator turns a normal function (with a docstring + type hints) into
something the model can request by name. The docstring is what the model
reads to decide *when* to use it — so write it like an instruction.

In [ ]:
@tool
def word_count(text: str) -> int:
    """Count the number of words in the given text. Use this whenever the user
    asks how many words are in something."""
    return len(text.split())

@tool
def reverse_string(text: str) -> str:
    """Reverse the characters in the given text."""
    return text[::-1]

tools = [word_count, reverse_string]
{t.name: t.description for t in tools}

## 2. Bind the tools to the model

`bind_tools` tells the model which tools exist. Now when it answers, it can
either reply normally **or** emit a `tool_calls` request instead of text.
llama.cpp supports this through the same OpenAI function-calling format your
proxy already forwards.

In [ ]:
model = ChatOpenAI(base_url='http://127.0.0.1:8080/v1', api_key='not-needed',
                   model='qwen3-14b-q4_k_m', temperature=0)
model_with_tools = model.bind_tools(tools)

## 3. The state and the nodes

Same `messages` state as Lesson 4. Two nodes:
- `call_model` — asks the (tool-aware) model.
- `tools` — a prebuilt `ToolNode` that runs whatever tool the model asked for
  and appends the result as a message. (You could hand-write this; the
  prebuilt just saves boilerplate.)

In [ ]:
class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def call_model(state: State) -> dict:
    return {'messages': [model_with_tools.invoke(state['messages'])]}

tool_node = ToolNode(tools)

## 4. The conditional edge — the whole point

After `call_model` runs, this router looks at the last message. If the model
asked for a tool, go to the `tools` node; otherwise we're done, go to `END`.
The router returns a **string naming the next node**.

In [ ]:
def route(state: State) -> str:
    last = state['messages'][-1]
    # A tool request shows up as .tool_calls on the AI message
    if getattr(last, 'tool_calls', None):
        return 'tools'
    return END

## 5. Wire it — note the loop

`add_conditional_edges` plugs the router in. The key line is the last edge:
`tools -> call_model`. That's what makes it a **loop** — after a tool runs, we
go back to the model so it can use the result (and maybe call another tool).

In [ ]:
builder = StateGraph(State)
builder.add_node('call_model', call_model)
builder.add_node('tools', tool_node)

builder.add_edge(START, 'call_model')
builder.add_conditional_edges('call_model', route)  # the diamond
builder.add_edge('tools', 'call_model')             # loop back

agent = builder.compile()
print(agent.get_graph().draw_ascii())

## 6. Run it

Ask something that needs a tool. Watch the message list grow: your question →
the model's tool request → the tool result → the model's final answer. That's
two trips through `call_model`, with the `tools` node in between.

In [ ]:
q = 'How many words are in this sentence: the quick brown fox jumps?'
out = agent.invoke({'messages': [HumanMessage(content=q)]})
for m in out['messages']:
    print(f'[{m.type}] {getattr(m, "content", "")!r}  tool_calls={getattr(m, "tool_calls", None)}')

Now ask something that needs **no** tool — the router sends it straight to
`END` after one model call, no loop.

In [ ]:
out = agent.invoke({'messages': [HumanMessage(content='Say hello in French.')]})
print(out['messages'][-1].content)

---
**What you just built** is a minimal *agent*: a model in a loop with tools,
where the model drives control flow. LangGraph even ships a one-liner for
this exact pattern — `from langgraph.prebuilt import create_react_agent` —
but building it by hand once is how you understand what that one-liner hides.